<a href="https://colab.research.google.com/github/ketanasindhusomisetty/GENAI_2520030487/blob/main/SKILL/Skill7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Experiment 7: Next-token probabilities - inspect logits and top-k candidates from a small open model (GPT-2)
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

def top_k_next(prompt, k=5):
    ids = tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(ids).logits[0, -1]          # scores for the NEXT token only
    probs = torch.softmax(logits, dim=-1)          # logits -> probability distribution
    p, idx = probs.topk(k)
    return [{"prompt": prompt, "rank": r + 1, "token": repr(tok.decode(i)), "token_id": int(i),
             "logit": round(float(logits[i]), 2), "probability": round(float(pp), 4)} for r, (pp, i) in enumerate(zip(p, idx))]

prompts = ["The capital of India is", "Machine learning is a subset of", "I drank a cup of", "The opposite of hot is"]
table = pd.DataFrame([row for pr in prompts for row in top_k_next(pr)])
print("Probability table (top-5 next tokens):")
print(table.to_string(index=False))

# Autoregressive generation, one token at a time (greedy) - shows that generation = repeated next-token prediction
ids = tok("Generative AI is", return_tensors="pt").input_ids
for step in range(8):
    with torch.no_grad():
        nxt = model(ids).logits[0, -1].argmax()
    ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    print(f"step {step + 1}: {tok.decode(ids[0])!r}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Probability table (top-5 next tokens):
                         prompt  rank         token  token_id  logit  probability
        The capital of India is     1        ' the'       262 -97.05       0.1131
        The capital of India is     2          ' a'       257 -97.72       0.0577
        The capital of India is     3       ' home'      1363 -98.05       0.0416
        The capital of India is     4      ' India'      3794 -98.06       0.0411
        The capital of India is     5        ' now'       783 -98.28       0.0332
Machine learning is a subset of     1        ' the'       262 -94.54       0.1260
Machine learning is a subset of     2    ' machine'      4572 -95.22       0.0637
Machine learning is a subset of     3     ' neural'     17019 -95.37       0.0550
Machine learning is a subset of     4       ' deep'      2769 -96.14       0.0255
Machine learning is a subset of     5 ' artificial'     11666 -96.49       0.0180
               I drank a cup of     1     ' coffee'      68